# 5. 더 깊은 CNN과 ResNet 이해하기

이 노트북은 `04_CNN_컬러_이미지_CIFAR10.ipynb` 다음 단계로, **왜 CNN을 더 깊게 쌓는지**, 그리고 **ResNet이 깊은 네트워크 학습을 어떻게 더 쉽게 만드는지**를 CIFAR-10 예제로 확인하는 실습입니다.

이전 노트북에서는 비교적 단순한 `ColorCNN`으로도 컬러 이미지 분류가 가능하다는 점을 봤습니다. 이제는 한 단계 더 나아가, 층을 더 깊게 쌓은 일반 CNN과 **skip connection(지름길 연결)** 을 가진 ResNet 계열 구조를 비교해 보겠습니다.


## 5-1. 왜 더 깊은 CNN이 필요할까?

컬러 이미지 문제에서는 단순한 경계선, 색 대비, 질감만 보는 것으로는 부족한 경우가 많습니다. 모델이 더 깊어지면 앞쪽 층에서 뽑은 간단한 특징들을 뒤쪽 층에서 여러 번 조합해, 더 복잡한 물체 개념을 표현할 수 있습니다.

하지만 층을 무작정 깊게 쌓는다고 항상 좋아지지는 않습니다. 실제로는 다음과 같은 문제가 생길 수 있습니다.

- gradient가 뒤쪽까지 잘 전달되지 않아 학습이 어려워질 수 있습니다.
- 네트워크가 깊어질수록 최적화가 까다로워질 수 있습니다.
- 단순히 층만 늘린 모델은 오히려 성능이 잘 안 오를 수도 있습니다.

ResNet은 이 지점에서 중요한 아이디어를 제시합니다. 바로 **입력을 몇 개 층 뒤로 직접 더해 주는 residual connection** 입니다.


## 5-2. Residual connection의 핵심 아이디어

일반적인 블록은 `입력 x -> 여러 층 통과 -> 출력 F(x)` 흐름입니다. ResNet 블록은 여기에 원래 입력 `x`를 다시 더해 `F(x) + x`를 만듭니다.

이 구조의 핵심 장점은 다음과 같습니다.

- 블록이 꼭 완전히 새로운 표현을 만들지 못해도, 최소한 기존 입력을 그대로 전달하는 경로가 남습니다.
- gradient가 더 직접적으로 흐를 수 있어 깊은 네트워크 학습이 안정적입니다.
- 층을 추가하더라도 "최소한 원래 정보는 유지"하는 방향으로 학습하기 쉬워집니다.

즉, ResNet은 "깊은 모델" 자체보다도 **깊은 모델을 실제로 학습 가능하게 만든 구조** 라고 볼 수 있습니다.


In [ ]:
# 필요 라이브러리가 없다면 아래 주석을 해제해서 설치하세요.
# !pip install torch torchvision matplotlib


In [ ]:
import copy
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('사용 장치:', device)


## 5-3. 데이터 준비

이번에도 `CIFAR-10`을 사용합니다. 데이터셋은 그대로 유지하고, 모델 구조만 더 깊게 바꾸어 보는 것이 핵심입니다.


In [ ]:
mean = (0.4914, 0.4822, 0.4465)
std = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

full_train_aug = datasets.CIFAR10(root='./data', train=True, download=True, transform=train_transform)
full_train_eval = datasets.CIFAR10(root='./data', train=True, download=False, transform=eval_transform)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=eval_transform)

classes = full_train_aug.classes
train_size = 45000
val_size = 5000

generator = torch.Generator().manual_seed(42)
train_dataset, _ = random_split(full_train_aug, [train_size, val_size], generator=generator)
_, val_dataset = random_split(full_train_eval, [train_size, val_size], generator=generator)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

print('Classes:', classes)
print('Train samples:', len(train_dataset))
print('Validation samples:', len(val_dataset))
print('Test samples:', len(test_dataset))


In [ ]:
def denormalize(image):
    mean_tensor = torch.tensor(mean).view(3, 1, 1)
    std_tensor = torch.tensor(std).view(3, 1, 1)
    return (image.cpu() * std_tensor + mean_tensor).clamp(0, 1)

images, labels = next(iter(train_loader))

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for ax, image, label in zip(axes.flat, images[:8], labels[:8]):
    ax.imshow(denormalize(image).permute(1, 2, 0))
    ax.set_title(classes[label])
    ax.axis('off')
plt.tight_layout()
plt.show()


## 5-4. 비교할 두 모델

이번에는 비슷한 깊이를 가진 두 모델을 비교합니다.

- `PlainDeepCNN`: 층을 더 깊게 쌓은 일반 CNN
- `SmallResNet`: residual block을 사용한 작은 ResNet 스타일 CNN

두 모델 모두 CIFAR-10에 맞춰 설계했지만, 핵심 차이는 **skip connection의 유무** 입니다.


In [ ]:
class PlainBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU()
        )

    def forward(self, x):
        return self.block(x)


class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU()

        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        identity = self.shortcut(x)

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        out = out + identity
        out = self.relu(out)
        return out


In [ ]:
class PlainDeepCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU()
        )
        self.layers = nn.Sequential(
            PlainBlock(32, 32, stride=1),
            PlainBlock(32, 64, stride=2),
            PlainBlock(64, 64, stride=1),
            PlainBlock(64, 128, stride=2),
            PlainBlock(128, 128, stride=1)
        )
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.layers(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x


class SmallResNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU()
        )
        self.layers = nn.Sequential(
            ResidualBlock(32, 32, stride=1),
            ResidualBlock(32, 64, stride=2),
            ResidualBlock(64, 64, stride=1),
            ResidualBlock(64, 128, stride=2),
            ResidualBlock(128, 128, stride=1)
        )
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.layers(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


plain_model = PlainDeepCNN().to(device)
resnet_model = SmallResNet().to(device)

print('PlainDeepCNN')
print(plain_model)
print('Trainable parameters:', f"{count_parameters(plain_model):,}")
print()
print('SmallResNet')
print(resnet_model)
print('Trainable parameters:', f"{count_parameters(resnet_model):,}")


Residual block에서는 출력이 단순히 `F(x)`가 아니라 `F(x) + x` 형태가 됩니다. 채널 수나 spatial size가 달라지는 구간에서는 `1x1 convolution`으로 shortcut 크기를 맞춰 준 뒤 더합니다.


## 5-5. 모델 학습 함수

두 모델을 같은 방식으로 학습시켜 validation / test 성능을 비교합니다. CPU만 사용할 때는 오래 걸릴 수 있으니 처음에는 `epochs = 3` 정도로 시작하는 것이 좋습니다. GPU를 쓸 수 있다면 `5 ~ 10 epoch`로 늘려 보세요.


In [ ]:
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * labels.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return total_loss / total, correct / total


def train_model(model, train_loader, val_loader, epochs=3, lr=0.001):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    best_state = copy.deepcopy(model.state_dict())
    best_val_acc = 0.0
    history = []

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * labels.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        train_loss = running_loss / total
        train_acc = correct / total
        val_loss, val_acc = evaluate(model, val_loader, criterion)

        history.append((train_loss, train_acc, val_loss, val_acc))
        print(f'Epoch {epoch + 1}/{epochs} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}')

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    return history, criterion


In [ ]:
print('=== PlainDeepCNN 학습 ===')
plain_history, plain_criterion = train_model(plain_model, train_loader, val_loader, epochs=3, lr=0.001)
plain_test_loss, plain_test_acc = evaluate(plain_model, test_loader, plain_criterion)
print('PlainDeepCNN Test Loss:', round(plain_test_loss, 4), '| Test Acc:', round(plain_test_acc, 4))

print()
print('=== SmallResNet 학습 ===')
resnet_history, resnet_criterion = train_model(resnet_model, train_loader, val_loader, epochs=3, lr=0.001)
resnet_test_loss, resnet_test_acc = evaluate(resnet_model, test_loader, resnet_criterion)
print('SmallResNet  Test Loss:', round(resnet_test_loss, 4), '| Test Acc:', round(resnet_test_acc, 4))


In [ ]:
epochs = range(1, len(plain_history) + 1)

plain_train_losses = [x[0] for x in plain_history]
plain_train_accs = [x[1] for x in plain_history]
plain_val_losses = [x[2] for x in plain_history]
plain_val_accs = [x[3] for x in plain_history]

resnet_train_losses = [x[0] for x in resnet_history]
resnet_train_accs = [x[1] for x in resnet_history]
resnet_val_losses = [x[2] for x in resnet_history]
resnet_val_accs = [x[3] for x in resnet_history]

plt.figure(figsize=(14, 4))

plt.subplot(1, 3, 1)
plt.plot(epochs, plain_train_losses, marker='o', label='Plain Train Loss')
plt.plot(epochs, plain_val_losses, marker='o', label='Plain Val Loss')
plt.plot(epochs, resnet_train_losses, marker='s', label='ResNet Train Loss')
plt.plot(epochs, resnet_val_losses, marker='s', label='ResNet Val Loss')
plt.title('Loss 비교')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 3, 2)
plt.plot(epochs, plain_train_accs, marker='o', label='Plain Train Acc')
plt.plot(epochs, plain_val_accs, marker='o', label='Plain Val Acc')
plt.plot(epochs, resnet_train_accs, marker='s', label='ResNet Train Acc')
plt.plot(epochs, resnet_val_accs, marker='s', label='ResNet Val Acc')
plt.title('Accuracy 비교')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 3, 3)
plt.bar(['Plain', 'ResNet'], [plain_test_acc, resnet_test_acc], color=['#94a3b8', '#2563eb'])
plt.ylim(0, 1)
plt.title('Test Accuracy')
plt.ylabel('Accuracy')

plt.tight_layout()
plt.show()


일반적으로는 같은 깊이라도 `SmallResNet`이 더 안정적으로 학습되거나, 더 적은 epoch에서도 validation/test 성능이 조금 더 잘 나오는 경우가 많습니다. 물론 짧은 학습에서는 차이가 작을 수도 있지만, 네트워크가 더 깊어질수록 residual connection의 장점이 더 분명해집니다.


## 5-6. 예측 결과 확인

마지막으로 ResNet 스타일 모델이 테스트 이미지에서 어떤 예측을 하는지 직접 확인합니다.


In [ ]:
resnet_model.eval()
images, labels = next(iter(test_loader))
images = images.to(device)
labels = labels.to(device)

with torch.no_grad():
    outputs = resnet_model(images)
    preds = outputs.argmax(dim=1)

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for ax, image, label, pred in zip(axes.flat, images[:8], labels[:8], preds[:8]):
    ax.imshow(denormalize(image).permute(1, 2, 0))
    ax.set_title(f'T: {classes[label]}\nP: {classes[pred]}')
    ax.axis('off')
plt.tight_layout()
plt.show()


## 정리

이번 노트북의 핵심은 다음과 같습니다.

- 컬러 이미지처럼 복잡한 문제에서는 더 깊은 CNN이 유리할 수 있습니다.
- 하지만 층을 단순히 깊게 쌓은 모델은 최적화가 까다롭고 학습이 잘 안 될 수 있습니다.
- ResNet은 `skip connection`을 통해 입력 정보를 직접 전달하고, gradient 흐름을 더 안정적으로 만들어 깊은 모델 학습을 쉽게 합니다.
- 실제 실습에서도 비슷한 깊이라면 residual block을 쓴 모델이 더 안정적인 성능을 보이는 경우가 많습니다.

다음 단계에서는 ResNet 계열을 더 확장해 보거나, `torchvision.models.resnet18` 같은 표준 모델을 가져와 transfer learning으로 연결해 볼 수 있습니다.
